In [1]:
import os
import torch
import pandas as pd

from tqdm import tqdm

from stock_bpt import StockBPT, LinearModel, NaiveModel
from dataloader_builder import build_dataloaders
from setup import StockBPT_cfg, LinearModel_cfg, NaiveModel_cfg
from setup import path_data_preprocessor
from model_training import model_setup, train_model_cuda

from model_training import train_model_cuda, evaluate_model, evaluate_best_model
from model_analysis import test_model, print_loss_analysis, process_losses, format_num, process_result, store_result

In [2]:
cuda = True if torch.cuda.is_available() else False

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.13.0+cu132
CUDA build: 13.2
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


## MODEL TRAINING ---------------------------

In [3]:
torch.manual_seed(1234)
dls, train_norms = build_dataloaders(path_data_preprocessor)

Building DataLoaders...


In [4]:
optimizer_data = [torch.optim.AdamW, 0.0004, 0.1]
scaler_data = [torch.amp.GradScaler, "cuda"]

max_epochs = 10

eval_bs = 1000

stockBPT, stockBPT_params, opt1, sca1, sch1 = model_setup(StockBPT, StockBPT_cfg, train_norms, device,
                                                *optimizer_data, *scaler_data)
linearModel, linearModel_params, opt2, sca2, sch2 = model_setup(LinearModel, LinearModel_cfg, train_norms, device, 
                                                      *optimizer_data, *scaler_data)
naiveModel = NaiveModel(NaiveModel_cfg, train_norms)
naiveModel.to(device)

Input Norm: torch.Size([12])|torch.Size([12])
Target Norm: torch.Size([4])|torch.Size([4])
3168256
5376


NaiveModel()

In [5]:
model_train_losses, model_val_losses = train_model_cuda(stockBPT, device, opt1, sca1, sch1, max_epochs, 
                                                        dls["train"], dls["val"], eval_bs)
linear_train_losses, linear_val_losses = train_model_cuda(linearModel, device, opt2, sca2, sch2, max_epochs,
                                                        dls["train"], dls["val"], eval_bs)


|          | 0.0% (00:00) Setting up...                                                                   

Epoch 1:

Learning Rate: 4.00e-04



|█         | 10.0% (01:58) Evaluating model on validation data... (670/671) [5510/55100]:                 

Epoch 1:
Training Loss:
   (MAE) 0.005358953028917313
   (NLL) -3.177783489227295
Validation Loss:
   (MAE) 0.006009611301124096
   (NLL) -2.104451894760132

Best Validation: -2.104451894760132
----------------------------------------------------------------------------------------------------

Learning Rate: 4.00e-04



|██        | 20.0% (03:52) Evaluating model on validation data... (670/671) [11020/55100]: 

...

|████████  | 80.0% (08:31) Evaluating model on validation data... (670/671) [44080/55100]: 

Epoch 8:
Training Loss:
   (MAE) 0.010974446311593056
   (NLL) -2.4431395530700684
Validation Loss:
   (MAE) 0.015234161168336868
   (NLL) -2.495615243911743

Best Validation: -2.9853882789611816
----------------------------------------------------------------------------------------------------

Learning Rate: 2.00e-04



|█████████ | 90.0% (09:30) Evaluating model on validation data... (670/671) [49590/55100]: 

Epoch 9:
Training Loss:
   (MAE) 0.004620499908924103
   (NLL) -2.973534107208252
Validation Loss:
   (MAE) 0.005573272705078125
   (NLL) -3.0160553455352783

Best Validation: -3.0160553455352783
----------------------------------------------------------------------------------------------------

Learning Rate: 2.00e-04



Epoch 10:
Training Loss:
   (MAE) 0.0064798821695148945
   (NLL) -2.8114261627197266
Validation Loss:
   (MAE) 0.009292766451835632
   (NLL) -2.711888313293457

Best Validation: -3.0160553455352783
----------------------------------------------------------------------------------------------------

Finished


## Model Analysis -------------------------

In [6]:
#* REUSES OBJETCS FROM TRAINING
analysis_steps = min(eval_bs, len(dls["train"])) + min(eval_bs, len(dls["val"])) + min(eval_bs, len(dls["test"]))
analysis_pbar = tqdm(total=3*analysis_steps, desc=f"Evaluating the best model parameters...".ljust(80),
                bar_format="|{bar}| {percentage:3.1f}% ({elapsed}) {desc}", position=0, leave=False)

#* Reevaluates models by their best parameters on train and val dataloaders
naive_losses = evaluate_model(dls["train"], dls["val"], naiveModel, device, eval_bs, analysis_pbar)
linear_losses = evaluate_best_model(linearModel, device, opt2, sca2, sch2, dls["train"], dls["val"], eval_bs, analysis_pbar, True)
bpt_losses = evaluate_best_model(stockBPT, device, opt1, sca1, sch1, dls["train"], dls["val"], eval_bs, analysis_pbar, True) 

#* Final evaluation on unseen test dataloader
naive_test_losses = test_model(dls["test"], naiveModel, device, eval_bs, analysis_pbar)
linear_test_losses = test_model(dls["test"], linearModel, device, eval_bs, analysis_pbar)
bpt_test_losses = test_model(dls["test"], stockBPT, device, eval_bs, analysis_pbar)


|██▋       | 26.4% (00:18) Evaluating model on training data... (14/1000) [1686/6378]:                    

[] []


|█████▏    | 52.4% (00:37) Evaluating model on validation data... (670/671) [3342/6378]: 

[] []


|██████████| 100.0% (01:17) Evaluating model on testing data... (454/455) [6378/6378]:   

In [7]:
for key, features in [("NLL", StockBPT_cfg["target_features"]),
                      ("STD", [f"{feature}_std" for feature in StockBPT_cfg["target_features"]]),
                      ("MAE", StockBPT_cfg["target_features"]),
                      ("Z^2", StockBPT_cfg["target_features"])]:
    print_loss_analysis(process_losses(bpt_losses + bpt_test_losses +
                                       linear_losses + linear_test_losses +
                                       naive_losses + naive_test_losses, key), 
                                       [stockBPT.cfg["name"], linearModel.cfg["name"], naiveModel.cfg["name"]],
                                       [format_num(stockBPT_params), format_num(linearModel_params), "0"], 
                                       features, key)


--------------------------------------------------------------------------------------------------------------

NLL

--------------------------------------------------------------------------------------------------------------

                    o        h        l        c        
StockGPT-B15: 3.2M
    Training:       -3.6346  -3.5646  -3.5959  -3.5836    >  -3.5947
    Validation:     -3.3968  -3.2374  -3.3831  -3.3602    >  -3.3444
    Testing:        -3.7049  -3.6381  -3.6541  -3.6622    >  -3.6648
    
LinearModel-B15: 5.4K
    Training:       -3.0496  -2.8289  -3.0719  -2.9437    >  -2.9735
    Validation:     -3.0968  -2.9837  -3.0507  -2.9330    >  -3.0161
    Testing:        -3.0956  -2.9809  -3.0565  -2.9378    >  -3.0177
    
NaiveModel-B15: 0
    Training:       2.7715   2.7754   2.7675   2.7717     >  2.7715
    Validation:     25.4199  25.4087  25.4303  25.4232    >  25.4205
    Testing:        3.2698   3.2713   3.2683   3.2701     >  3.2699
    

-------------------

In [8]:
return

import importlib
import setup
importlib.reload(setup)
from setup import PATH_RESULTS_RESIDUALS

store_result(PATH_RESULTS_RESIDUALS, process_result(stockBPT, bpt_losses, bpt_test_losses, max_epochs))
store_result(PATH_RESULTS_RESIDUALS, process_result(linearModel, linear_losses, linear_test_losses, max_epochs))
store_result(PATH_RESULTS_RESIDUALS, process_result(naiveModel, naive_losses, naive_test_losses, max_epochs))

print(pd.read_parquet(PATH_RESULTS_RESIDUALS))

SyntaxError: 'return' outside function (3832739517.py, line 1)

In [ ]:
from data_scrapper import scrape_data, get_all_tickers
from data_filler import fill_data
from data_preprocessor import preprocess_data
from setup import API_KEY, TIMEFRAME
import pandas_market_calendars as mcal

In [ ]:
from setup import ID
all_tickers = get_all_tickers("raw_data/all_tickers_trimmed_1_30", API_KEY)
scrape_data(API_KEY,
              f"raw_data/data_{ID}min_2026",
              250,
              all_tickers,
              mcal.get_calendar("NYSE").schedule("2026-01-01","2026-8-1").index,
              TIMEFRAME)
fill_data(f"raw_data/data_{ID}min_2026",
          f"filled_raw_data/data_{ID}min_2026",
          mcal.get_calendar("NYSE").schedule("2026-01-01","2026-8-1").index)
preprocess_data(f"filled_raw_data/data_{ID}min_2026",
                f"preprocessed_data/data_{ID}min_2026",
                mcal.get_calendar("NYSE").schedule("2026-01-01","2026-8-1").index, [0.75, 0.9])

|██████████| 100.0% (13:03) Data fetching completed
|          | 0.0% (00:45) Saving file for batch_036...: 
|██████████| 100.0% (00:45) Saving preprocessed_data\data_15min_2026\test\batch_036.arrow...                : 


In [9]:
dls, train_norms = build_dataloaders("preprocessed_data/data_1min_2026")

|██████████| 100.0% (01:32) Evaluating model on testing data... (454/455) [6378/6378]: 

Building DataLoaders...


In [10]:
test_losses = test_model(dls["test"], stockBPT, device, eval_bs)
print(test_losses)

({'NLL': tensor([13.4641,  9.7023,  9.6980,  9.5874], device='cuda:0'), 'STD': tensor([0.0163, 0.0173, 0.0176, 0.0176], device='cuda:0'), 'MAE': tensor([0.0448, 0.0437, 0.0428, 0.0404], device='cuda:0'), 'Z^2': tensor([33.5915, 25.9467, 25.9065, 25.6912], device='cuda:0')},)
